In [50]:
# Install packages
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns

#Set file paths
root_fldr = Path.cwd().parent
raw_fldr = root_fldr/"data"/"raw"
processed_fldr = root_fldr/'data'/'processed'

#Load calflora data and convert coordinates to points
calflora1_raw = pd.read_csv(raw_fldr/"calflora-out.csv", encoding="cp1252")
calflora2_raw = pd.read_csv(raw_fldr/"calflora-out part 2.csv", encoding="cp1252")
calflora1 = gpd.GeoDataFrame(calflora1_raw, geometry=gpd.points_from_xy(calflora1_raw["longitude"], calflora1_raw["latitude"]),
    crs="EPSG:4326"
)
calflora2 = gpd.GeoDataFrame(calflora2_raw, geometry=gpd.points_from_xy(calflora2_raw["longitude"], calflora2_raw["latitude"]),
    crs="EPSG:4326"
)

calflora_full = pd.concat([calflora1, calflora2])

#Drop overlapped observation from final dataset
calflora_full = calflora_full.drop_duplicates()
calflora_full["date"] = pd.to_datetime(calflora_full['date'], format = '%m/%d/%Y')

In [56]:
calflora_full_cleaned = (
    calflora_full
    .drop(columns=["quality", "county"])
    .set_crs("EPSG:4326", allow_override=True)
            .to_crs("EPSG:3310")
    .assign(
            year=lambda df: df["date"].dt.year.astype(int),
            date=lambda df: df["date"].dt.tz_localize(None))
)

In [57]:
calflora_full_cleaned.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 215256 entries, 0 to 65446
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                215256 non-null  str           
 1   common_name       214951 non-null  str           
 2   scientific_name   215256 non-null  str           
 3   date              215256 non-null  datetime64[us]
 4   number_of_plants  176576 non-null  str           
 5   latitude          215256 non-null  float64       
 6   longitude         215256 non-null  float64       
 7   geometry          215256 non-null  geometry      
 8   year              215256 non-null  int64         
dtypes: datetime64[us](1), float64(2), geometry(1), int64(1), str(4)
memory usage: 16.4 MB


In [60]:
#save to processed folder
calflora_full_cleaned.to_file(
    processed_fldr / "calflora_full_cleaned.gpkg",
    driver="GPKG"
)

In [59]:
calflora_full_cleaned["year"].value_counts()

year
2021    33367
2019    30009
2022    27959
2020    26305
2023    26044
2024    25305
2025    23816
2026    22451
Name: count, dtype: int64